In [ ]:
import cv2
import mediapipe as mp
import time
import random


# 创建手势识别器
BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
RunningMode = mp.tasks.vision.RunningMode

options = GestureRecognizerOptions(
    base_options=BaseOptions(
        model_asset_path="gesture_recognizer.task"
    ),
    running_mode=RunningMode.VIDEO
)

recognizer = GestureRecognizer.create_from_options(options)


# 打开摄像头
camera = cv2.VideoCapture(0)

start_time = time.time()


# 手势对应关系
gesture_map = {
    "Closed_Fist": "ROCK",
    "Open_Palm": "PAPER",
    "Victory": "SCISSORS"
}


# 这一轮是否结束
round_done = False


while True:

    success, frame = camera.read()

    if not success:
        break


    # OpenCV BGR → RGB
    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    # 转成 MediaPipe 图片
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    # timestamp
    timestamp_ms = int(
        (time.time() - start_time) * 1000
    )


    # 识别手势
    result = recognizer.recognize_for_video(
        mp_image,
        timestamp_ms
    )


    # 如果识别到手势，而且这一轮还没结束
    if result.gestures and not round_done:

        gesture = result.gestures[0][0]
        gesture_name = gesture.category_name


        # 只接受石头剪刀布这三个动作
        if gesture_name in gesture_map:

            player = gesture_map[gesture_name]

            computer = random.choice([
                "ROCK",
                "PAPER",
                "SCISSORS"
            ])


            # 判断输赢
            if player == computer:

                result_text = "DRAW"

            elif (
                (player == "ROCK" and computer == "SCISSORS")
                or
                (player == "PAPER" and computer == "ROCK")
                or
                (player == "SCISSORS" and computer == "PAPER")
            ):

                result_text = "YOU WIN"

            else:

                result_text = "YOU LOSE"


            print("YOU:", player)
            print("AI:", computer)
            print(result_text)


            # 锁定这一轮
            round_done = True


    # 显示摄像头
    cv2.imshow(
        "Gesture Recognition",
        frame
    )


    # 按 Q 退出
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


camera.release()
recognizer.close()
cv2.destroyAllWindows()

YOU: PAPER
AI: ROCK
YOU WIN


KeyboardInterrupt: 

: 